In [ ]:
# if you run it on your .ven env and you have already installed the dependencies, you can ignore this cell
!pip install --user -U processmining pm4py

In [2]:
from importlib.metadata import version
version("processmining")

'1.0.19'

In [3]:
"""
Experiment runner for BPIC OCEL logs (with per-unfold-call logging).

Now loops n from 2 to 10, reads logs from ./data,
prints progress for every step, and records read-time per run.
"""

from __future__ import annotations

import time
from pathlib import Path
from typing import List, Dict, Any

import numpy as np
import pandas as pd
import pm4py
import processmining.log.ocel as ocelhelper

In [ ]:
# -----------------------------
# Parameters
# -----------------------------
# Please note that BPIC 2016 is very large, and if your machine does not have enough memory, you may want to limit the experiment to other years.
YEARS = [2014, 2016, 2017, 2019]
N_VALUES = range(2, 11)   # n = 2..10

summary_csv_path = Path("bpic_dummy_drill_unfold_summary.csv")
calls_csv_path   = Path("bpic_dummy_drill_unfold_calls.csv")

In [5]:
# -----------------------------
# Helpers
# -----------------------------
def safe_len(x) -> int:
    try:
        return int(len(x))
    except Exception:
        return 0


def split_group_factory(n: int):
    """Return a split_group function that uses given n."""
    def split_group(g: pd.DataFrame) -> pd.Series:
        idx = g.index
        if len(g) < n:
            cats = (np.arange(len(g)) % n) + 1
            return pd.Series(cats, index=idx)

        cats = pd.qcut(
            np.arange(len(g)),
            q=n,
            labels=range(1, n + 1),
        )
        return pd.Series(cats, index=idx)

    return split_group


# -----------------------------
# One-year experiment
# -----------------------------
def run_for_year(year: int, n: int):

    print(f"[INFO] Loading OCEL for year {year} with n={n}...")

    # FIX: use short year (14, 16, 17, 19)
    short = str(year)[-2:]
    file_name = f"./data/BPIC{short}.jsonocel"

    t0 = time.perf_counter()
    ocel = pm4py.read.read_ocel_json(file_name)
    read_time_s = time.perf_counter() - t0
    print(f"[OK] Loaded {file_name} in {read_time_s:.2f}s")

    # --- Structural metrics ---
    n_events = safe_len(ocel.events)
    n_objects = safe_len(ocel.objects)
    n_relations = safe_len(ocel.relations)
    n_event_types = ocel.events["ocel:activity"].nunique(dropna=True)
    n_object_types = ocel.objects["ocel:type"].nunique(dropna=True)

    # --- Most common object type ---
    obj_type_counts = ocel.objects["ocel:type"].value_counts()
    most_common_obj_type = obj_type_counts.idxmax()
    most_common_obj_type_count = int(obj_type_counts.max())

    # --- Most common event type for that object type ---
    rel_subset = ocel.relations[ocel.relations["ocel:type"] == most_common_obj_type]
    most_common_event_type = rel_subset["ocel:activity"].value_counts().idxmax()
    most_common_event_type_count = int(
        len(
            ocel.relations[
                (ocel.relations["ocel:type"] == most_common_obj_type)
                & (ocel.relations["ocel:activity"] == most_common_event_type)
            ]
        )
    )

    # --- Dummy drilling ---
    split_group = split_group_factory(n)
    cats = ocel.objects.groupby("ocel:type").apply(split_group)

    if isinstance(cats.index, pd.MultiIndex):
        cats.index = cats.index.get_level_values(-1)

    ocel.objects["dummy_drilling"] = cats.reindex(ocel.objects.index)

    # --- Drill-down ---
    print(f"[INFO] Drill-down for year {year}, n={n}...")
    t0 = time.perf_counter()
    drilled_ocel = ocelhelper.drill_down(
        ocel,
        most_common_obj_type,
        "dummy_drilling",
    )
    drill_time_s = time.perf_counter() - t0
    print(f"[OK] Drill-down completed in {drill_time_s:.2f}s")

    # --- Unfold calls ---
    print(f"[INFO] Unfolding {n} categories for year {year}...")
    unfold_total_s = 0.0
    unfold_calls = 0
    call_rows: List[Dict[str, Any]] = []

    _unfolded = drilled_ocel
    for i in range(1, n + 1):
        unfold_key = f"({most_common_obj_type},{i})"
        t0 = time.perf_counter()
        _unfolded = ocelhelper.unfold(
            _unfolded,
            most_common_event_type,
            unfold_key,
        )
        dt = time.perf_counter() - t0
        unfold_total_s += dt
        unfold_calls += 1

        print(f"    - Unfold call {i}/{n} completed in {dt:.4f}s")

        call_rows.append({
            "Year": year,
            "File": file_name,
            "n_categories": n,
            "Unfold call idx": i,
            "Unfold key": unfold_key,
            "Unfold event type": most_common_event_type,
            "Unfold object type": most_common_obj_type,
            "Unfold dt (s)": dt,
            "Read time (s)": read_time_s,
            "Drilldown time (s)": drill_time_s,
        })

    print(f"[DONE] Year {year}, n={n}: total unfold time {unfold_total_s:.2f}s\n")

    unfold_avg_per_call_s = unfold_total_s / unfold_calls if unfold_calls else 0.0

    # --- Summary row ---
    summary_row = {
        "Year": year,
        "File": file_name,
        "n_categories": n,
        "Read time (s)": read_time_s,
        "Drilldown time (s)": drill_time_s,
        "Unfold total time (s)": unfold_total_s,
        "Unfold calls (count)": unfold_calls,
        "Unfold avg per call (s)": unfold_avg_per_call_s,
        "Num events": n_events,
        "Num objects": n_objects,
        "Num relations": n_relations,
        "Num event types": n_event_types,
        "Num object types": n_object_types,
        "Most common object type": most_common_obj_type,
        "Most common object type count": most_common_obj_type_count,
        "Most common event type (for that obj type)": most_common_event_type,
        "Most common event type count (for that obj type)": most_common_event_type_count,
    }

    return summary_row, call_rows

In [6]:
# -----------------------------
# Main
# -----------------------------
def main(years: List[int], n_values, summary_csv: Path, calls_csv: Path):

    summary_rows: List[Dict[str, Any]] = []
    all_call_rows: List[Dict[str, Any]] = []

    for n in n_values:
        print("\n==============================")
        print(f"Running experiments for n={n}")
        print("==============================")

        for year in years:
            print(f"\n--- Starting year {year} (n={n}) ---")
            summary_row, call_rows = run_for_year(year, n)
            summary_rows.append(summary_row)
            all_call_rows.extend(call_rows)
            print(f"--- Finished year {year} (n={n}) ---\n")

        print(f"[COMPLETE] All years done for n={n}\n")

    df_summary_run = pd.DataFrame(summary_rows)
    df_calls_run = pd.DataFrame(all_call_rows)

    # --- Append summary ---
    if summary_csv.exists():
        df_summary_existing = pd.read_csv(summary_csv)
        df_summary_all = pd.concat(
            [df_summary_existing, df_summary_run],
            ignore_index=True
        )
    else:
        df_summary_all = df_summary_run

    df_summary_all.to_csv(summary_csv, index=False)

    # --- Append calls ---
    if calls_csv.exists():
        df_calls_existing = pd.read_csv(calls_csv)
        df_calls_all = pd.concat(
            [df_calls_existing, df_calls_run],
            ignore_index=True
        )
    else:
        df_calls_all = df_calls_run

    df_calls_all.to_csv(calls_csv, index=False)

    print(f"[SAVED] Summary → {summary_csv}")
    print(f"[SAVED] Calls   → {calls_csv}")


if __name__ == "__main__":
    main(YEARS, N_VALUES, summary_csv_path, calls_csv_path)


Running experiments for n=2

--- Starting year 2014 (n=2) ---
[INFO] Loading OCEL for year 2014 with n=2...


/cephyr/users/aminj/Alvis/.local/lib/python3.8/site-packages/pm4py/util/dt_parsing/parser.py:82: UserWarning: ISO8601 strings are not fully supported with strpfromiso for Python versions below 3.11
  warnings.warn(


[OK] Loaded ./data/BPIC14.jsonocel in 35.54s
[INFO] Drill-down for year 2014, n=2...
[OK] Drill-down completed in 3.68s
[INFO] Unfolding 2 categories for year 2014...
    - Unfold call 1/2 completed in 1.6345s
    - Unfold call 2/2 completed in 1.2220s
[DONE] Year 2014, n=2: total unfold time 2.86s

--- Finished year 2014 (n=2) ---


--- Starting year 2016 (n=2) ---
[INFO] Loading OCEL for year 2016 with n=2...
[OK] Loaded ./data/BPIC16.jsonocel in 700.31s
[INFO] Drill-down for year 2016, n=2...
[OK] Drill-down completed in 68.63s
[INFO] Unfolding 2 categories for year 2016...
    - Unfold call 1/2 completed in 28.0594s
    - Unfold call 2/2 completed in 31.4966s
[DONE] Year 2016, n=2: total unfold time 59.56s

--- Finished year 2016 (n=2) ---


--- Starting year 2017 (n=2) ---
[INFO] Loading OCEL for year 2017 with n=2...
[OK] Loaded ./data/BPIC17.jsonocel in 49.52s
[INFO] Drill-down for year 2017, n=2...
[OK] Drill-down completed in 2.95s
[INFO] Unfolding 2 categories for year 2017..